# CI/CD for ML with GitHub Actions

> **Track:** ML Engineer | **Level:** Intermediate | **Time:** 2-3 hours

## Overview
In traditional software, CI/CD runs tests and deploys code. In ML, CI/CD also validates **data**, runs **model quality gates**, manages the **model registry**, and deploys model artifacts. This notebook teaches you to build a production-grade ML CI/CD pipeline.

### What You'll Learn
- CI/CD concepts applied to ML
- Data validation with Pandera (schema enforcement)
- Automated ML testing with pytest
- GitHub Actions workflow for ML pipelines
- Model quality gates before registry promotion
- Staging → production promotion

```bash
pip install pandera pytest mlflow scikit-learn pandas numpy pyyaml
```

In [ ]:
# Setup: create a sample ML project structure
import pandas as pd
import numpy as np
import json
import yaml
from pathlib import Path
from typing import Any

np.random.seed(42)

# Create project directory structure
dirs = ["ml_project/data", "ml_project/models", "ml_project/tests", "ml_project/.github/workflows"]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

# Generate sample training data (churn prediction)
n = 1000
raw_data = pd.DataFrame({
    "customer_id": range(n),
    "tenure_months": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(20, 120, n), 2),
    "num_products": np.random.randint(1, 5, n),
    "support_tickets": np.random.randint(0, 10, n),
    "churn": np.random.choice([0, 1], n, p=[0.75, 0.25])
})
raw_data.to_csv("ml_project/data/churn_train.csv", index=False)

print("Project structure created.")
print(f"Training data: {raw_data.shape} | Churn rate: {raw_data['churn'].mean():.1%}")

## 1. Data Validation with Pandera

The first CI step: validate that incoming data conforms to the expected schema. Catch data pipeline bugs before they corrupt model training.

In [ ]:
# Data validation schema with Pandera
try:
    import pandera as pa
    from pandera import Column, Check, DataFrameSchema

    churn_schema = DataFrameSchema(
        columns={
            "customer_id": Column(int, Check.ge(0)),
            "tenure_months": Column(int, Check.between(0, 120)),
            "monthly_charges": Column(float, Check.between(0, 500)),
            "num_products": Column(int, Check.between(1, 10)),
            "support_tickets": Column(int, Check.ge(0)),
            "churn": Column(int, Check.isin([0, 1])),
        },
        checks=[
            Check(lambda df: df.shape[0] > 100, error="Dataset too small"),
            Check(lambda df: df["churn"].mean() < 0.9, error="Churn rate suspiciously high"),
        ]
    )

    # Validate the training data
    validated_data = churn_schema.validate(raw_data)
    print("✓ Data validation passed!")
    print(f"  Rows validated: {len(validated_data)}")

    # Test with bad data
    bad_data = raw_data.copy()
    bad_data.loc[0, "churn"] = 99  # invalid label
    try:
        churn_schema.validate(bad_data)
    except pa.errors.SchemaError as e:
        print(f"✓ Bad data caught: {str(e)[:80]}...")

except ImportError:
    print("Pandera not installed. pip install pandera")
    print("Showing schema definition only.")

## 2. ML Unit Tests with pytest

ML tests differ from software tests: we test model properties (not just code correctness), data invariants, and prediction behavior.

In [ ]:
# Write ML test suite (to be run with pytest)
ml_tests_code = '''
# ml_project/tests/test_model.py
import pytest
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

FEATURES = ["tenure_months", "monthly_charges", "num_products", "support_tickets"]
MIN_ACCURACY = 0.70
MIN_AUC = 0.72

@pytest.fixture
def training_data():
    return pd.read_csv("data/churn_train.csv")

@pytest.fixture
def trained_model(training_data):
    X = training_data[FEATURES]
    y = training_data["churn"]
    model = GradientBoostingClassifier(n_estimators=50, random_state=42)
    model.fit(X, y)
    return model, X, y

def test_model_accuracy(trained_model):
    model, X, y = trained_model
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    assert scores.mean() >= MIN_ACCURACY, f"Accuracy {scores.mean():.3f} < {MIN_ACCURACY}"

def test_model_no_nan_predictions(trained_model):
    model, X, _ = trained_model
    preds = model.predict_proba(X)
    assert not np.isnan(preds).any(), "Model produces NaN predictions"

def test_prediction_range(trained_model):
    model, X, _ = trained_model
    probs = model.predict_proba(X)[:, 1]
    assert (probs >= 0).all() and (probs <= 1).all(), "Probabilities out of [0,1] range"

def test_model_not_trivially_predicting(trained_model):
    model, X, _ = trained_model
    preds = model.predict(X)
    assert len(np.unique(preds)) > 1, "Model predicts only one class"
'''

with open("ml_project/tests/test_model.py", "w") as f:
    f.write(ml_tests_code)

print("ML test suite written to ml_project/tests/test_model.py")
print()
print("Run with: cd ml_project && pytest tests/ -v")

## 3. GitHub Actions Workflow for ML

The GitHub Actions workflow orchestrates the entire ML CI/CD pipeline on every push or PR.

In [ ]:
# Generate the GitHub Actions ML pipeline YAML
ml_pipeline_yaml = {
    "name": "ML Pipeline CI/CD",
    "on": {"push": {"branches": ["main", "develop"]}, "pull_request": {"branches": ["main"]}},
    "jobs": {
        "validate-data": {
            "runs-on": "ubuntu-latest",
            "steps": [
                {"uses": "actions/checkout@v4"},
                {"uses": "actions/setup-python@v4", "with": {"python-version": "3.11"}},
                {"run": "pip install pandera pandas"},
                {"name": "Validate training data schema", "run": "python scripts/validate_data.py"},
            ]
        },
        "train-and-test": {
            "runs-on": "ubuntu-latest",
            "needs": "validate-data",
            "steps": [
                {"uses": "actions/checkout@v4"},
                {"run": "pip install scikit-learn mlflow pytest pandas numpy"},
                {"name": "Train model", "run": "python scripts/train.py"},
                {"name": "Run ML tests", "run": "pytest tests/ -v --tb=short"},
                {"name": "Check model quality gate", "run": "python scripts/quality_gate.py"},
            ]
        },
        "register-and-deploy": {
            "runs-on": "ubuntu-latest",
            "needs": "train-and-test",
            "if": "github.ref == 'refs/heads/main'",
            "steps": [
                {"name": "Register model to staging", "run": "python scripts/register_model.py --stage Staging"},
                {"name": "Run integration tests", "run": "pytest tests/integration/ -v"},
                {"name": "Promote to production", "run": "python scripts/register_model.py --stage Production"},
            ]
        }
    }
}

workflow_path = Path("ml_project/.github/workflows/ml_pipeline.yml")
with open(workflow_path, "w") as f:
    yaml.dump(ml_pipeline_yaml, f, default_flow_style=False, sort_keys=False)

print("GitHub Actions workflow written to:")
print(f"  {workflow_path}")
print()
with open(workflow_path) as f:
    print(f.read()[:600], "...")

## 4. Model Quality Gates

A **quality gate** blocks deployment if the model doesn't meet minimum performance thresholds. This prevents regressions from reaching production.

In [ ]:
# Implement a model quality gate
import sys
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

QUALITY_GATES: dict[str, float] = {
    "min_accuracy": 0.70,
    "min_auc_roc": 0.72,
    "max_training_time_seconds": 60,
}

def run_quality_gate(model, X: pd.DataFrame, y: pd.Series) -> dict[str, Any]:
    """Evaluate model against quality gates. Returns pass/fail per gate."""
    import time
    results: dict[str, Any] = {}

    # Gate 1: Accuracy
    acc_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    results["accuracy"] = {
        "value": round(acc_scores.mean(), 4),
        "threshold": QUALITY_GATES["min_accuracy"],
        "passed": acc_scores.mean() >= QUALITY_GATES["min_accuracy"]
    }

    # Gate 2: AUC-ROC
    auc_scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
    results["auc_roc"] = {
        "value": round(auc_scores.mean(), 4),
        "threshold": QUALITY_GATES["min_auc_roc"],
        "passed": auc_scores.mean() >= QUALITY_GATES["min_auc_roc"]
    }

    results["overall_passed"] = all(g["passed"] for g in results.values() if isinstance(g, dict))
    return results

# Train model and run gates
FEATURES = ["tenure_months", "monthly_charges", "num_products", "support_tickets"]
X = raw_data[FEATURES]
y = raw_data["churn"]

model = GradientBoostingClassifier(n_estimators=50, random_state=42)
model.fit(X, y)

gate_results = run_quality_gate(model, X, y)
print("=== MODEL QUALITY GATE RESULTS ===")
for gate, result in gate_results.items():
    if isinstance(result, dict):
        status = "✓ PASS" if result["passed"] else "✗ FAIL"
        print(f"{status} | {gate}: {result['value']} (threshold: {result['threshold']})")
print()
print(f"OVERALL: {'APPROVED FOR DEPLOYMENT ✓' if gate_results['overall_passed'] else 'BLOCKED — DO NOT DEPLOY ✗'}")

## Exercises

1. **Add a data drift gate**: Extend `run_quality_gate` to also compare the training data distribution against a reference dataset using KL divergence or Kolmogorov-Smirnov test. Block deployment if feature distributions shift significantly.

2. **Implement model versioning**: Write a `register_model.py` script that logs the model to MLflow, tags it with `git_commit_sha`, `accuracy`, `auc_roc`, and `training_date`, and transitions the previous staging model to 'Archived' before promoting the new one.

3. **Write the GitHub Actions workflow locally**: Use the `act` tool (https://github.com/nektos/act) to run GitHub Actions workflows locally. Trigger your ML pipeline YAML and observe how the jobs chain together with the `needs` dependency.